In [1]:
import MDAnalysis as mda
import mdtraj as md
import os, sys, glob
import numpy as np

from openmm import *
from openmm.app import *
from openmm.unit import *

#sys.path.append('/media/volume/Josephs-Volume/githubs/Bridgeport/')

import ForceFields.ForceFieldHandler as ff_handler
from Ligand.Ligand import Ligand
from ForceFields.OpenMMJoiner import Joiner

from MotorRow import MotorRow

In [2]:
#Run MotorRow to get a benchmark
#MR = MotorRow(pdb_file=os.path.join(os.getcwd(), 'test_data','RS_sets','RS1125.pdb'),
#             system_xml=os.path.join(os.getcwd(), 'test_data','RS_sets','RS1125_sys.xml'),
#             working_directory=os.path.join(os.getcwd(), 'Static_Env','MotorRowBenchMark'))

In [3]:
#state_fn, pdb_fn = MR.main(os.path.join(os.getcwd(), 'test_data','RS_sets','RS1125.pdb'))

In [4]:
class IO_Grid:
  """
  Class to read and write alchemical grids.

  Data is a dictionary with
  spacing - the grid spacing, in Angstroms.
  counts - the number of points in each dimension.
  vals - the values.
  All are numpy arrays.
  """
  def __init__(self):
    pass

  def read(self, FN, multiplier=None):
    """
    Reads a grid in dx or netcdf format
    The multiplier affects the origin and spacing.
    """
    if FN is None:
      raise Exception('File is not defined')
    elif FN.endswith('.dx') or FN.endswith('.dx.gz'):
      data = self._read_dx(FN)
    elif FN.endswith('.nc'):
      data = self._read_nc(FN)
    else:
      raise Exception('File type not supported')
    if multiplier is not None:
      data['origin'] = multiplier * data['origin']
      data['spacing'] = multiplier * data['spacing']
    return data

  def _read_dx(self, FN):
    """
    Reads a grid in dx format
    """
    if FN.endswith('.dx'):
      F = open(FN, 'r')
    else:
      import gzip
      F = gzip.open(FN, 'r')

    # Read the header
    line = F.readline()
    while line.find('object') == -1:
      line = F.readline()
    header = {}
    header['counts'] = [int(x) for x in line.split(' ')[-3:]]
    for name in ['origin', 'd0', 'd1', 'd2']:
      header[name] = [float(x) for x in F.readline().split(' ')[-3:]]
    F.readline()
    header['npts'] = int(F.readline().split(' ')[-3])

    # Test to make sure the grid type is okay.
    # These conditions are not absolultely essential,
    #   but they reduce the number of subtraction operations.
    if not (header['d0'][1] == 0 and header['d0'][2] == 0
            and header['d1'][0] == 0 and header['d1'][2] == 0
            and header['d2'][0] == 0 and header['d2'][1] == 0):
      raise Exception('Trilinear grid must be in original basis')
    if not (header['d0'][0] > 0 and header['d1'][1] > 0
            and header['d2'][2] > 0):
      raise Exception('Trilinear grid must have positive coordinates')

    # Read the data
    vals = np.ndarray(shape=header['npts'], dtype=float)
    index = 0
    while index < header['npts']:
      line = F.readline()[:-1]
      items = [float(item) for item in line.split()]
      vals[index:index + len(items)] = items
      index = index + len(items)
    F.close()

    data = {
      'origin':np.array(header['origin']), \
      'spacing':np.array([header['d0'][0],header['d1'][1],header['d2'][2]]), \
      'counts':np.array(header['counts']), \
      'vals':vals}
    return data

  def _read_nc(self, FN):
    """
    Reads a grid in netcdf format
    """
    from netCDF4 import Dataset
    grid_nc = Dataset(FN, 'r')
    data = {}
    for key in list(grid_nc.variables):
      data[key] = np.array(grid_nc.variables[key][:][0][:])
    grid_nc.close()
    return data

  def write(self, FN, data, multiplier=None):
    """
    Writes a grid in dx or netcdf format.
    The multiplier affects the origin and spacing.

    """
    if multiplier is not None:
      data_n = {
        'origin': multiplier * data['origin'],
        'counts': data['counts'],
        'spacing': multiplier * data['spacing'],
        'vals': data['vals']
      }
    else:
      data_n = data
    if FN.endswith('.nc'):
      self._write_nc(FN, data_n)
    elif FN.endswith('.dx') or FN.endswith('.dx.gz'):
      self._write_dx(FN, data_n)
    else:
      raise Exception('File type not supported')

  def _write_dx(self, FN, data):
    """
    Writes a grid in dx format
    """
    n_points = data['counts'][0] * data['counts'][1] * data['counts'][2]
    if FN.endswith('.dx'):
      F = open(FN, 'w')
    else:
      import gzip
      F = gzip.open(FN, 'w')

    F.write("""object 1 class gridpositions counts {0[0]} {0[1]} {0[2]}
origin {1[0]} {1[1]} {1[2]}
delta {2[0]} 0.0 0.0
delta 0.0 {2[1]} 0.0
delta 0.0 0.0 {2[2]}
object 2 class gridconnections counts {0[0]} {0[1]} {0[2]}
object 3 class array type double rank 0 items {3} data follows
""".format(data['counts'], data['origin'], data['spacing'], n_points))

    for start_n in range(0, len(data['vals']), 3):
      F.write(' '.join(['%6e' % c
                        for c in data['vals'][start_n:start_n + 3]]) + '\n')

    F.write('object 4 class field\n')
    F.write('component "positions" value 1\n')
    F.write('component "connections" value 2\n')
    F.write('component "data" value 3\n')
    F.close()

  def _write_nc(self, FN, data):
    """
    Writes a grid in netcdf format
    """
    n_points = data['counts'][0] * data['counts'][1] * data['counts'][2]
    from netCDF4 import Dataset
    grid_nc = Dataset(FN, 'w', format='NETCDF4')
    grid_nc.createDimension('one', 1)
    grid_nc.createDimension('n_cartesian', 3)
    grid_nc.createDimension('n_points', n_points)
    grid_nc.createVariable('origin', 'f8', ('one', 'n_cartesian'))
    grid_nc.createVariable('counts', 'i8', ('one', 'n_cartesian'))
    grid_nc.createVariable('spacing', 'f8', ('one', 'n_cartesian'))
    grid_nc.createVariable('vals', 'f8', ('one', 'n_points'), zlib=True)
    for key in data.keys():
      grid_nc.variables[key][:] = data[key]
    grid_nc.close()

  def truncate(self, in_FN, out_FN, counts, multiplier=None, in_xyz=None, out_xyz=None):
    """
    Truncates the grid at the origin and
    with a limited number of counts per dimension

    multiplier is for the values, not the grid scaling
    """
    data_o = self.read(in_FN)

    if (in_xyz is None and out_xyz is None):
        nyz_o = data_o['counts'][1] * data_o['counts'][2]
        nz_o = data_o['counts'][2]
    
        print('origin', data_o['origin'])
        print('spacing', data_o['spacing'])
    
        min_i = int(-data_o['origin'][0] / data_o['spacing'][0])
        min_j = int(-data_o['origin'][1] / data_o['spacing'][1])
        min_k = int(-data_o['origin'][2] / data_o['spacing'][2])
    
        print('min_i', min_i)
        print('min_j', min_j)
        print('min_k', min_k)
        print('vals.shape', data_o['vals'].shape)
    
        # vals = np.ndarray(shape=tuple(counts), dtype=float)
        # print 'vals.shape', vals.shape 
        # for i in range(counts[0]):
        #   for j in range(counts[1]):
        #     for k in range(counts[2]):
        #       # print 'i,j,k', i, j, k, (i+min_i)*nyz_o + (j+min_j)*nz_o + (k+min_k)
        #       vals[i,j,k] = data_o['vals'][(i+min_i)*nyz_o + (j+min_j)*nz_o + (k+min_k)]

        vals = np.array([[[data_o['vals'][(i + min_i) * nyz_o + (j + min_j) * nz_o + (k + min_k)] for k in range(counts[2]) ] for j in range(counts[1])] for i in range(counts[0])])

    else:
        matching_inds = np.empty(out_xyz.shape[0], dtype=int)
        for i, (x, y, z) in enumerate(out_xyz):
            same_x_inds = np.where(in_xyz[:,0] == x)[0]
            same_y_inds = np.where(in_xyz[:,1] == y)[0]
            same_z_inds = np.where(in_xyz[:,2] == z)[0]
            xy_ind = np.intersect1d(same_x_inds, same_y_inds)
            ind = np.intersect1d(same_z_inds, xy_ind)
            assert len(ind) == 1
            matching_inds[i] = ind[0]

        vals = data_o['vals'][matching_inds]
        

    if multiplier is not None:
      vals = vals * multiplier

    if out_xyz is not None:
        data_n = {'origin': out_xyz[0], \
          'counts':counts, 'spacing':data_o['spacing'], 'vals':vals.flatten()}
    else:
        data_n = {'origin': np.array([0,0,0]), \
          'counts':counts, 'spacing':data_o['spacing'], 'vals':vals.flatten()}
    self.write(out_FN, data_n)

In [5]:
def select_whole_residues(universe, ref_selection):
        """
        universe = mdanalysis universe to select from
        atom_selection = mdanalysis selection string for included atoms

        returns group_included, group_excluded
        
        Retrieve whole residues for a selection
        """
        group1 = universe.select_atoms(ref_selection)
        group2 = universe.select_atoms('same residue as group group1', group1=group1)
        group3 = universe.select_atoms('not same residue as group group2', group2=group2)
        return group2, group3

def parameterize_from_pdb(pdb_file, lig_smiles, nonbondedMethod=CutoffNonPeriodic, nonbondedCutoff=1.0*nanometer,
                          lig_resname='UNK', cleanup=True, build_w_obc2=False):

    #Seperate the stuff
    uni = mda.Universe(pdb_file)
    print(len(uni.atoms))
    not_lig = uni.select_atoms(f'not resname {lig_resname}')
    lig = uni.select_atoms('resname UNK')

    if not os.path.isdir('./temp/'):
        os.mkdir('./temp/')
    
    not_lig.write('./temp/dummy_P.pdb')
    lig.write('./temp/dummy_L.pdb')

    #Get the Parameters
    ff_xmls = ['amber14/protein.ff14SB.xml',
               'amber14/lipid17.xml',
               '/media/volume/Josephs-Volume/githubs/Bridgeport/ForceFields/wat_opc3.xml']

    #if build_w_obc2:
    #    ff_xmls.append('implicit/obc2.xml')
    
    ff = ForceField(*ff_xmls)
    pdb = PDBFile('./temp/dummy_P.pdb')
    rec_top, rec_pos = pdb.getTopology(), pdb.getPositions()
    rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff)
    
    #Ligand Too
    ligand = Ligand(working_dir='./temp/', name='dummy_L', resname='UNK', smiles=lig_smiles)
    ligand.prepare_ligand()
    lig_sys, lig_top, lig_pos = ff_handler.ForceFieldHandler('./temp/dummy_L.sdf').main(use_pme='NoCutoff')
    
    #Joiner
    sys, top, pos = Joiner((lig_sys, lig_top, lig_pos), (rec_sys, rec_top, rec_pos)).main()
    print(sys.getNumParticles())
    for force in sys.getForces():
        if hasattr(force, 'getNonbondedMethod'):
            print(force)
            print(force.getNonbondedMethod(), force.getNumParticles())
    
    if cleanup:
        os.remove('./temp/dummy_P.pdb')
        os.remove('./temp/dummy_L.pdb')
        os.remove('./temp/dummy_L.sdf')
        os.rmdir('./temp')
    
    return sys, top, pos

def simulate_from_filepair(pdb_top_fn, sys_xml_fn,
                           temp=300, ts=0.001, n_steps=1e6, #1ns
                           dcd_fn='output.dcd', dcd_freq=1000,
                           stdout_fn='output.stdout', stdout_freq=1000,
                           working_dir='./'):
    if not os.path.isdir(working_dir):
        os.makedirs(working_dir, exist_ok=True)
    #Construct
    pdb = PDBFile(pdb_top_fn)
    with open(sys_xml_fn, 'r') as f:
        system = XmlSerializer.deserialize(f.read())
    integrator = LangevinMiddleIntegrator(temp*kelvin, 1/picosecond, ts*picoseconds)
    sim = Simulation(pdb.topology, system, integrator)
    _ = sim.context.setPositions(pdb.positions)
    #Minimize
    _ = sim.minimizeEnergy()
    with open(os.path.join(working_dir,'minimized.pdb'), 'w') as f:
        _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    #Simulate
    _ = sim.reporters.append(DCDReporter(os.path.join(working_dir, dcd_fn), dcd_freq))
    _ = sim.reporters.append(StateDataReporter(os.path.join(working_dir, stdout_fn), stdout_freq, step=True, potentialEnergy=True, temperature=True, speed=True))
    _ = sim.step(n_steps)

    #Final
    state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
    contents = XmlSerializer.serialize(state)
    with open(os.path.join(working_dir, 'final_state.xml'), 'w') as f:
        _ = f.write(contents)
    with open(os.path.join(working_dir,'final_top.pdb'), 'w') as f:
        _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    return None

In [6]:
pdb_fn = 'Static_Env/MotorRowBenchMark/Step_5_wrapped.pdb'
#ligand_smiles = "CCN(CC)C(=O)N[C@@H]1CN([C@@H]2Cc3c[nH]c4c3c(ccc4)C2=C1)C"
ligand_smiles = "[H][C@]1(C[C@@H]2[C@H]3C(=C[C@]4(CCCC[C@@]24[H])O[C@@]13C)C1=NCCO1)C1=CC=CC(C)=N1"
pdb = PDBFile(pdb_fn)
u = mda.Universe(pdb_fn)

In [7]:
envelope, outside = select_whole_residues(u, 'protein or resname UNK or (around 5 (protein or resname UNK))')

In [8]:
envelope, outside, envelope.indices.shape, outside.indices.shape

(<AtomGroup with 15783 atoms>,
 <AtomGroup with 29852 atoms>,
 (15783,),
 (29852,))

#### Just simulate the rigid envelope

In [9]:
system, top, pos = parameterize_from_pdb('Static_Env/MotorRowBenchMark/Step_5_wrapped.pdb',
                                         lig_smiles=ligand_smiles)

45635


/home/exouser/miniconda3/envs/chimpss/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


03/08/2026 20:13:26//Saved prepared ligand to ./temp/dummy_L.pdb ./temp/dummy_L.sdf


/home/exouser/miniconda3/envs/chimpss/lib/python3.12/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/media/volume/Large-Storage/githubs/ChiMPSS/ForceFields/ForceFieldHandler.py:158: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  positions = np.array(self.interchange.positions) * nanometer


45635
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x749d2e31ccb0> >
1 45635


In [ ]:
for i in range(system.getNumParticles()):
    if i in outside.indices:
        system.setParticleMass(i, 0)

In [ ]:
with open('test.pdb', 'w') as f:
   _ = PDBFile.writeFile(top, pos, file=f, keepIds=True)
with open('test.xml', 'w') as f:
    f.write(XmlSerializer.serialize(system))

In [ ]:
simulate_from_filepair('test.pdb', 'test.xml')

#### Load the bridgeport system and make the rigid atoms rigid - should be PME?

In [11]:
with open('/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.xml', 'r') as f:
    bridgeport_sys = XmlSerializer.deserialize(f.read())

for i in range(bridgeport_sys.getNumParticles()):
    if i in outside.indices:
        bridgeport_sys.setParticleMass(i, 0)

with open('test.pdb', 'w') as f:
    _ = PDBFile.writeFile(pdb.topology, pdb.positions, file=f, keepIds=True)

with open('test.xml', 'w') as f:
    f.write(XmlSerializer.serialize(bridgeport_sys))

In [ ]:
simulate_from_filepair('test.pdb', 'test.xml')

#### Manipulate the system forces

To prepare for adding GridForces, only add parameters for particles in the ENVELOPE

In [9]:
ref_sys, top, pos = parameterize_from_pdb('Static_Env/MotorRowBenchMark/Step_5_wrapped.pdb')

75735


/home/exouser/miniconda3/envs/prep/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


12/16/2025 22:12:41//Saved prepared ligand to ./temp/dummy_L.pdb ./temp/dummy_L.sdf


[22:12:41] WARNING: More than one matching pattern found - picking one

/media/volume/Josephs-Volume/githubs/Bridgeport/ForceFields/ForceFieldHandler.py:158: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  positions = np.array(self.interchange.positions) * nanometer


75735
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7849c1a9d3f0> >
1 75735


In [9]:
#Edit the particles
for i in range(ref_sys.getNumParticles()):
    if i in outside.indices:
        ref_sys.setParticleMass(i, 0)

In [10]:
#Bonds
hbf = [force for force in ref_sys.getForces() if type(force) == HarmonicBondForce][0]
bond_force = HarmonicBondForce()
for i in range(hbf.getNumBonds()):
    params = hbf.getBondParameters(i)
    if params[0] in envelope.indices or params[1] in envelope.indices:
        bond_force.addBond(*params)
ref_sys.removeForce([index for index, force in enumerate(ref_sys.getForces()) if type(force)==HarmonicBondForce][0])
ref_sys.addForce(bond_force)

4

In [12]:
ref_sys.getForces()

[<openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x7f50c93bdef0> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7f51c8c8ceb0> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x7f50c93bd670> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x7f51c8c8ce30> >,
 <openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x7f51c8c8ea30> >]

In [13]:
#Angles
haf = [force for force in ref_sys.getForces() if type(force) == HarmonicAngleForce][0]
angle_force = HarmonicAngleForce()
for i in range(haf.getNumAngles()):
    params = haf.getAngleParameters(i)
    if params[0] in envelope.indices or params[1] in envelope.indices or params[2] in envelope.indices:
        angle_force.addAngle(*params)
ref_sys.removeForce([index for index, force in enumerate(ref_sys.getForces()) if type(force)==HarmonicAngleForce][0])
ref_sys.addForce(angle_force)

4

In [14]:
ref_sys.getForces()

[<openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x7f51c8c8fe70> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7f51e8bb78f0> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x7f51c8c8e8b0> >,
 <openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x7f51c8c8e630> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x7f51c8c8fab0> >]

In [15]:
#Torsions
ptf = [force for force in ref_sys.getForces() if type(force) == PeriodicTorsionForce][0]
torsion_force = PeriodicTorsionForce()
for i in range(ptf.getNumTorsions()):
    params = ptf.getTorsionParameters(i)
    if params[0] in envelope.indices or params[1] in envelope.indices or params[2] in envelope.indices or params[3] in envelope.indices:
        torsion_force.addTorsion(*params)
ref_sys.removeForce([index for index, force in enumerate(ref_sys.getForces()) if type(force)==PeriodicTorsionForce][0])
ref_sys.addForce(torsion_force)

4

In [16]:
ref_sys.getForces()

[<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7f51c8c8cb70> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x7f51c8c8f5b0> >,
 <openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x7f51c8c8ed30> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x7f50ca82d130> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x7f51c8c8ffb0> >]

In [17]:
nbf = [force for force in ref_sys.getForces() if type(force) == NonbondedForce][0]
cutoff = 1.2 *nanometer #nanometers

eps_solvent = nbf.getReactionFieldDielectric()
krf = (1/ (cutoff**3)) * (eps_solvent - 1) / (2*eps_solvent + 1)
crf = (1/ cutoff) * (3* eps_solvent) / (2*eps_solvent + 1)

ONE_4PI_EPS0 = 138.935456
energy_expression  = "4*epsilon*((sigma/r)^12 - (sigma/r)^6) + ONE_4PI_EPS0*chargeprod*(1/r + krf*r*r - crf);"
energy_expression += "epsilon = sqrt(epsilon1*epsilon2);"
energy_expression += "sigma = 0.5*(sigma1+sigma2);"
energy_expression += "chargeprod = charge1*charge2;"
nonbonded_force = CustomNonbondedForce(energy_expression)

nonbonded_force.addGlobalParameter('krf', krf.value_in_unit(nanometer**-3))
nonbonded_force.addGlobalParameter('crf', crf.value_in_unit(nanometer**-1))
nonbonded_force.addGlobalParameter('ONE_4PI_EPS0', ONE_4PI_EPS0)

nonbonded_force.addPerParticleParameter('charge')
nonbonded_force.addPerParticleParameter('sigma')
nonbonded_force.addPerParticleParameter('epsilon')

for index in range(nbf.getNumParticles()):
    charge, sigma, epsilon = nbf.getParticleParameters(index)
    nonbonded_force.addParticle([charge, sigma, epsilon])
    
for index in range(nbf.getNumExceptions()):
    j, k, chargeprod, sigma, epsilon = nbf.getExceptionParameters(index)
    nonbonded_force.addExclusion(j, k)

nonbonded_force.addInteractionGroup(envelope.indices, envelope.indices)
nonbonded_force.addInteractionGroup(envelope.indices, outside.indices)
nonbonded_force.setNonbondedMethod(CustomNonbondedForce.NoCutoff)
#nonbonded_force.setCutoffDistance(cutoff)
ref_sys.removeForce([index for index, force in enumerate(ref_sys.getForces()) if type(force)==NonbondedForce][0])
ref_sys.addForce(nonbonded_force)

4

In [18]:
ref_sys.getForces()

[<openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x7f51c8f47630> >,
 <openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x7f51c8f46b70> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x7f51c8f469f0> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x7f51c8f45f70> >,
 <openmm.openmm.CustomNonbondedForce; proxy of <Swig Object of type 'OpenMM::CustomNonbondedForce *' at 0x7f51c8f464f0> >]

In [ ]:
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.001*picoseconds)
sim = Simulation(top, ref_sys, integrator)
_ = sim.context.setPositions(pos)
_ = sim.minimizeEnergy()
with open(os.path.join('./','minimized.pdb'), 'w') as f:
    _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)

In [ ]:
_ = sim.reporters.append(DCDReporter('output.dcd', 100))
_ = sim.reporters.append(StateDataReporter('output.stdout', 100, step=True, potentialEnergy=True, temperature=True, speed=True))

In [ ]:
#Minimize
#Simulate
_ = sim.step(1000000)

#Final
state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
contents = XmlSerializer.serialize(state)
with open('final_state.xml', 'w') as f:
    _ = f.write(contents)
with open('final_top.pdb', 'w') as f:
    _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)

### Just copy MotorRow

In [5]:
MR = MotorRow(pdb_file='/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.pdb',
             system_xml='/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.xml',
             working_directory='Static_Env/MotorRow_Static')

In [6]:
from datetime import datetime
from MotorRow_utils import *

In [ ]:
self = MR
state_in = '/media/volume/Josephs-Volume/githubs/MotorRow/Static_Env/MotorRowBenchMark/Step_5.xml'
positions_from_pdb = '/media/volume/Josephs-Volume/githubs/MotorRow/Static_Env/MotorRowBenchMark/Step_5.pdb'
state_xml_out = None
pdb_out = None
fc_pos = 300.0
nsteps = 500000
temp = 300.0
dt = 2.0
ncycles = 50
nstdout = 1000
fn_stdout = None
ndcd = 5000
append_dcd = False 
fn_dcd = None
press = 1.0 
restrain_lig = False

#Before ANY STEP
start = datetime.now()
#Establish State
with open(self.system_xml) as f:
    system = XmlSerializer.deserialize(f.read())
# Get atoms
assert positions_from_pdb is not None
crds, prt_heavy, mem_heavy, lig_heavy_atoms = get_positions_from_pdb(positions_from_pdb, lig_resname=self.lig_resname, lig_chain=self.lig_chain)

# Ligand Restraint
if restrain_lig:
    if len(lig_heavy_atoms) > 0:
        system = restrain_atoms(system, crds, np.array(lig_heavy_atoms)[:,0], rst_name='lig_k', rst_strength=86.68*(joule)/(angstrom*angstrom*mole))
    elif self.lig_resname is not None or self.lig_chain is not None:
        raise Exception('Could not find ligand...')   
        
system.addForce(MonteCarloBarostat(press*bar, temp*kelvin, 100))
pdb = PDBFile(positions_from_pdb)
u = mda.Universe(positions_from_pdb)
envelope, outside = select_whole_residues(u, 'protein or resname UNK or (around 2 (protein or resname UNK))')

#Edit the particles
for index in outside.indices:
    system.setParticleMass(index, 0)

# Any Step Establish Simulation
integrator = LangevinMiddleIntegrator(temp*kelvin, 1/picosecond, dt*femtosecond)
simulation = Simulation(self.topology, system, integrator)

simulation.loadState(state_in)

if fn_stdout is None:
    fn_stdout = os.path.join(self.abs_work_dir, f'step.stdout')

if fn_dcd is None:
    fn_dcd = os.path.join(self.abs_work_dir, f'step.dcd')

SDR = app.StateDataReporter(fn_stdout, nstdout, step=True, time=True,
                            potentialEnergy=True, temperature=True, progress=False,
                            remainingTime=True, speed=True, volume=True,
                            totalSteps=nsteps, separator=' : ')

simulation.reporters.append(SDR)
DCDR = app.DCDReporter(file=fn_dcd, reportInterval=ndcd, append=append_dcd, enforcePeriodicBox=True)
simulation.reporters.append(DCDR)
print(f'Starting Step with forces {simulation.system.getForces()}')
print(f'Starting Step with box_vectors {simulation.system.getDefaultPeriodicBoxVectors()}')

# Write out state.xml
if state_xml_out is None:
    state_xml_out = os.path.join(self.abs_work_dir, f'Step.xml')

# Determine no. of steps per cycle
steps_per_cycle = int(nsteps / ncycles)
              
# Reconfigure steps needed to take if appending
if append_dcd:
    steps_taken = float(simulation.context.getTime().value_in_unit(femtosecond) / dt)
    cycles_completed = math.floor(steps_taken / steps_per_cycle)
    print('steps_taken', steps_taken)
    print('cycles_complete', cycles_completed)
else:
    cycles_completed = 0
    simulation.context.setTime(0)

print('ncycles', ncycles)
print('nsteps', nsteps)
print('dt', dt)
print('steps_per_cycle', steps_per_cycle)
for cycle in range(cycles_completed+1, ncycles+1):
    print('Cycle', cycle, 'to', ((cycle/ncycles) * ((nsteps * dt) / 1e6)), 'ns')
    simulation.step(steps_per_cycle)
    self._describe_state(simulation, f'Step')
    self._write_state(simulation, state_xml_out)

end = datetime.now() - start
print(f'Step completed after {end}')
print(f'Box Vectors after this step {simulation.system.getDefaultPeriodicBoxVectors()}')

if pdb_out is None:
    pdb_out = os.path.join(self.abs_work_dir, f'Step_{stepnum}.pdb')
self._write_structure(simulation, pdb_out)

for i in range(3):
    print('########################################################################################')
print(state_xml_out, pdb_out)

### Try using grids

In [9]:
ref_sys, top, pos = parameterize_from_pdb('Static_Env/MotorRowBenchMark/Step_5_wrapped.pdb', lig_smiles=ligand_smiles)

45635


/home/exouser/miniconda3/envs/chimpss/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


03/08/2026 20:32:13//Saved prepared ligand to ./temp/dummy_L.pdb ./temp/dummy_L.sdf


/home/exouser/miniconda3/envs/chimpss/lib/python3.12/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/media/volume/Large-Storage/githubs/ChiMPSS/ForceFields/ForceFieldHandler.py:158: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  positions = np.array(self.interchange.positions) * nanometer


45635
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7d5aeba099b0> >
1 45635


In [10]:
from datetime import datetime
import jax
import jax.numpy as jnp

In [11]:
print(envelope.positions.max(axis=0), envelope.positions.min(axis=0))
print(outside.positions.max(axis=0), outside.positions.min(axis=0))

[74.539 66.35  94.519] [-0.858 -7.008 10.34 ]
[ 79.098  74.435 102.077] [-10.559  -9.374  -0.62 ]


In [12]:
indices, pos = jnp.array(outside.indices), jnp.array(outside.positions)

In [13]:
dim = pos.max(axis=0) - pos.min(axis=0) #size
spacing = 0.125 #Angstrom desired spacing
#dim *= 0.1 #Angstrom to nanometer if necessary
counts = jnp.ceil(dim/spacing).astype(int)
spacing = [spacing]*3

In [14]:
### Coordinates of grid points
print('Calculating grid coordinates')
startTime = datetime.now()

grid_x = np.zeros(shape=tuple(counts), dtype=float)
for j in range(counts[1]):
    for k in range(counts[2]):
          grid_x[:,j,k] = spacing[0]*np.arange(counts[0])

grid_y = np.zeros(shape=tuple(counts), dtype=float)
for i in range(counts[0]):
    for k in range(counts[2]):
          grid_y[i,:,k] = spacing[1]*np.arange(counts[1])

grid_z = np.zeros(shape=tuple(counts), dtype=float)
for i in range(counts[0]):
    for j in range(counts[1]):
          grid_x[i,j,:] = spacing[2]*np.arange(counts[2])

grid = {'x':jnp.array(grid_x),
        'y':jnp.array(grid_y),
        'z':jnp.array(grid_y)}

endTime = datetime.now()
print(f' in {endTime-startTime}')

Calculating grid coordinates



KeyboardInterrupt



In [15]:
### Coordinates of grid points
print('Calculating grid coordinates')
startTime = datetime.now()
# xyz = get_grid_pos(center, counts, spacing)
grid = {}
grid['x'] = np.zeros(shape=tuple(counts), dtype=float)
grid['y'] = np.zeros(shape=tuple(counts), dtype=float)
grid['z'] = np.zeros(shape=tuple(counts), dtype=float)
for i in range(counts[0]):
  for j in range(counts[1]):
    for k in range(counts[2]):
      grid['x'][i,j,k] = i*spacing[0]
      grid['y'][i,j,k] = j*spacing[1]
      grid['z'][i,j,k] = k*spacing[2]

for key in ['x', 'y', 'z']:
    grid[key] = jnp.array(grid[key])

endTime = datetime.now()
print(f' in {endTime-startTime}')

#These points are however relative to the origin...

Calculating grid coordinates
 in 0:08:32.562599


In [16]:
n_points = jnp.prod(counts)
n_atoms = indices.shape[0]

In [17]:
nbf = [force for force in ref_sys.getForces() if type(force) == NonbondedForce][0]
startTime = datetime.now()
print("Reading Parameters for Grid Particles")
charges = jnp.zeros(n_atoms) #elementary charge
depths = jnp.zeros(n_atoms) #OpenMM = kJ/mol convert to AMBER kcal/mol
radii = jnp.zeros(n_atoms) #OpenMM = nm convert to AMBER Angstrom
for i, atom_index in enumerate(indices):
    q, r, e = nbf.getParticleParameters(atom_index)
    charges = charges.at[i].set(q.value_in_unit(elementary_charge))
    depths = depths.at[i].set(e.value_in_unit(kilocalorie_per_mole))
    radii = radii.at[i].set(r.value_in_unit(angstrom))

root_depths = jnp.sqrt(depths)
diameters = 2*radii

endTime = datetime.now()
print(f' in {endTime-startTime}')

Reading Parameters for Grid Particles
 in 0:01:48.679895


In [19]:
points = np.array([grid['x'].flatten(), grid['y'].flatten(), grid['z'].flatten()]).T

2026-03-08 21:17:30.993454: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.47GiB (rounded to 1584086272)requested by op 
2026-03-08 21:17:30.993555: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ****************************************************************************************____________
E0308 21:17:30.993580 1279486 pjrt_stream_executor_client.cc:3026] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1584086064 bytes. [tf-allocator-allocation-error='']


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1584086064 bytes.

In [ ]:
print('Calculating Grid Potentials')
startTime = datetime.now()
def distance_all_to_point(p):
    return jnp.sqrt(jnp.sum((pos - p)**2, axis=-1))

coulomb_all_atom_to_point = jax.jit(lambda p: jnp.sum(332.06*charges/distance_all_to_point(p)))
LJR_all_atom_to_point = jax.jit(lambda p: jnp.sum(root_depths*(diameters**6)/(distance_all_to_point(p)**12)))
LJA_all_atom_to_point = jax.jit(lambda p: jnp.sum(-2*root_depths*(diameters**3)/(distance_all_to_point(p)**6)))

coulomb_flat = jnp.zeros(points.shape[0])
ljr_flat = jnp.zeros(points.shape[0])
lja_flat = jnp.zeros(points.shape[0])

n_reps = 10000 #Chosen to optimize speed, the number of iterations to perform parallel computation (here 10k sequential processes executing a parallel computaiton)

for i in range(n_reps):
    coulomb_flat = coulomb_flat.at[i::n_reps].set(jax.vmap(coulomb_all_atom_to_point, in_axes=(0))(points[i::n_reps]))
    ljr_flat = ljr_flat.at[i::n_reps].set(jax.vmap(LJR_all_atom_to_point, in_axes=(0))(points[i::n_reps]))
    lja_flat = lja_flat.at[i::n_reps].set(jax.vmap(LJA_all_atom_to_point, in_axes=(0))(points[i::n_reps]))

    if i % (n_reps//10) == 0:
        print(f"{100*i/n_reps:2.2f}% Done")
endTime = datetime.now()
print(f' in {endTime-startTime}')

In [ ]:
coulomb_flat.reshape(counts).shape, ljr_flat.reshape(counts).shape, lja_flat.reshape(counts).shape

In [ ]:
#Cap Lennard-Jones Potentials
u_max = 10000.0
ljr_flat = u_max*jnp.tanh(ljr_flat/u_max)
lja_flat = u_max*jnp.tanh(lja_flat/u_max)

In [ ]:
print('Saving...')
io = IO_Grid()
for fn in ['test_high_res_ele', 'test_high_res_ljr', 'test_high_res_lja']:
    if 'ele' in fn:
        data = np.array(coulomb_flat)
    elif 'ljr' in fn:
        data = np.array(ljr_flat)
    elif 'lja' in fn:
        data = np.array(lja_flat)
    else:
        raise Exception('HOW')

    for ext in ['.dx', '.nc']:
        save_name = fn+ext
        print(save_name)
        io.write(save_name, {'origin':np.array([0,0,0]),
                             'spacing':np.array(spacing),
                             'counts':np.array(counts),
                             'vals':data})
print("Done!")

#### Now actually put the grids into a force

And perform AMBER to OpenMM Conversions

In [9]:
envelope.write('test.pdb')
cut_sys, cut_top, cut_pos = parameterize_from_pdb('test.pdb')

/home/exouser/miniconda3/envs/prep/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


18500
12/18/2025 16:33:29//Saved prepared ligand to ./temp/dummy_L.pdb ./temp/dummy_L.sdf


[16:33:29] WARNING: More than one matching pattern found - picking one

/media/volume/Josephs-Volume/githubs/Bridgeport/ForceFields/ForceFieldHandler.py:158: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  positions = np.array(self.interchange.positions) * nanometer


18500
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7aefe76e63b0> >
1 18500


In [10]:
# from __future__ import print_function
# from openmm import app
# from openmm import unit
# from openmm.vec3 import Vec3
# import sys
# import gridforceplugin
# import numpy as np


def grid_read(FN):
    """
    Reads a grid in a netcdf format
    The multiplier affects the origin and spacing
    """
    if FN is None:
        raise Exception('File is not defined')
    elif FN.endswith('.nc'):
        from netCDF4 import Dataset
        grid_nc = Dataset(FN, 'r')
        data = {}
        for key in list(grid_nc.variables):
            data[key] = np.array(grid_nc.variables[key][:][0][:])
        grid_nc.close()
    else:
        raise Exception('File type not supported')
    return data

def getGridForce(FN, unit_conversion):
    data = grid_read(FN)
    force = gridforceplugin.GridForce()
    nx = int(data['counts'][0])
    ny = int(data['counts'][1])
    nz = int(data['counts'][2])
    force.addGridCounts(nx, ny, nz)
    data['spacing'] *= 0.1
    force.addGridSpacing(data['spacing'][0],
                        data['spacing'][1],
                        data['spacing'][2])
    data['vals'] *= unit_conversion
    for val in data['vals']:
        force.addGridValue(val)
    return force

In [11]:
initial_forces = cut_sys.getNumForces()
nbf = [force for force in cut_sys.getForces() if type(force) == NonbondedForce][0]
# Add forces
forces_to_add = [('test_ele.nc', 4.184),
                 ('test_ljr.nc', np.sqrt(4.184)*1.0e6),
                 ('test_lja.nc', np.sqrt(4.184)*1.0e3)]

#Accumulate charges, sigmas, epsilons
scaling_factors = []
for i in range(cut_sys.getNumParticles()):
    q, s, e = nbf.getParticleParameters(i)
    #This should be OpenMM Convention?? - Yes this should be OpenMM units
    scaling_factors.append([q.value_in_unit(elementary_charge),
                            s.value_in_unit(nanometer),
                            e.value_in_unit(kilojoule_per_mole)])

In [12]:
import gridforceplugin

for grid_file, unit_conv in forces_to_add:
    force = getGridForce(grid_file, unit_conv)
    if 'ele' in grid_file:
        for chg, _, _ in scaling_factors:
            force.addScalingFactor(chg)
    else:
        for _, rVdw, eps in scaling_factors:
            scale = np.sqrt(eps)*(2.0*rVdw)**(6 if 'ljr' in grid_file else 3)
            force.addScalingFactor(scale)
    cut_sys.addForce(force)

assert cut_sys.getNumForces() == initial_forces + 3

In [22]:
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.00005*picoseconds)
sim = Simulation(cut_top, cut_sys, integrator)
_ = sim.context.setPositions(cut_pos)
_ = sim.minimizeEnergy()
with open(os.path.join('./','minimized.pdb'), 'w') as f:
    _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)

In [23]:
_ = sim.reporters.append(DCDReporter('output.dcd', 2))
_ = sim.reporters.append(StateDataReporter('output.stdout', 2, step=True, potentialEnergy=True, temperature=True, speed=True))

In [24]:
cut_sys.getForces()

[<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x7aefe63bcef0> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x7aefe63bd2f0> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7aefe63bdab0> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x7aefe63bd430> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x7aefe63bd5b0> >,
 <openmm.openmm.Force; proxy of <Swig Object of type 'OpenMM::Force *' at 0x7aefe63bdaf0> >,
 <openmm.openmm.Force; proxy of <Swig Object of type 'OpenMM::Force *' at 0x7aefe63bda70> >,
 <openmm.openmm.Force; proxy of <Swig Object of type 'OpenMM::Force *' at 0x7aefe63bdef0> >]

In [25]:
#Minimize
#Simulate
_ = sim.step(1000000)

#Final
state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
contents = XmlSerializer.serialize(state)
with open('final_state.xml', 'w') as f:
    _ = f.write(contents)
with open('final_top.pdb', 'w') as f:
    _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)

ValueError: Particle position is NaN.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#nan

#### Calculate ele and Lennard-Jones potential energies at grid points
```
Units: kcal/mol A e

For ele potential:
E = 1/(4*pi*eps_o) q1 q2 / r

                      1 kg m^2  | (1.60217646E-19 e)^2 | cal    s^2   |   1 kcal | 1E10 A |
--------------------------------|----------------------|--------------|----------|--------| * 6.0221415E+23
4 pi 8.854187817629E-12 C^2 s^2 |                  C^2 | 4.184 kg m^2 | 1000 cal |  1 m   |

Prefactor is:
1/(4*math.pi*8.85418781762E-12)*(1.60217646E-19**2)/4.184*1E10*6.0221415E+23/1000 = 332.06 kcal/mol A e^2
```

In [ ]:
stor_dir = 'implicit_membrane_equil2_nocutoff'
if not os.path.isdir(stor_dir):
    os.makedirs(stor_dir, exist_ok=True)

name_dict = {'15A': 'protein or resname UNK or (around 15 (protein or resname UNK))',
             '10A': 'protein or resname UNK or (around 10 (protein or resname UNK))',
             '5A': 'protein or resname UNK or (around 5 (protein or resname UNK))',
             '2A': 'protein or resname UNK or (around 2 (protein or resname UNK))',
             'PL': 'protein or resname UNK'}
uni = mda.Universe(bridgeport_output_pdb)
for key, value in name_dict.items():
    group = select_whole_residues(uni, value)
    group.write(os.path.join(stor_dir, f"cut_{key}.pdb"))

In [ ]:
a

In [ ]:

class Static_Env_Simulation(MotorRow):
    def __init__(self, pdb_file, system_xml, working_directory, **kwargs):
        super.__init__(pdb_file, system_xml, working_directory, **kwargs))

        self.openmm_pdb = PDBFile(pdb_file)
        self.Universe = mda.Universe(pdb_file)
        with open(system_xml, 'r') as f:
            self.system = XmlSerializer.deserialize(f.read())

            

In [2]:
motor_row_final = '/media/volume/Josephs-Volume/githubs/MotorRow/implicit_membrane/MotorRowEquil/Step_5_wrapped.pdb'

In [3]:
pdb = PDBFile(motor_row_final)

In [13]:
def select_whole_residues(universe, selection_string):
    group1 = universe.select_atoms(selection_string)
    group2 = universe.select_atoms('same residue as group group1', group1=group1)
    group3 = universe.select_atoms('not same residue as group group2', group2=group2)
    return group2, group3

In [5]:
pos = pdb.getPositions(asNumpy=True)

In [6]:
traj = md.Trajectory(pos, md.Topology.from_openmm(pdb.topology))

In [7]:
pos = traj.xyz

In [8]:
import jax
import jax.numpy as jnp

In [9]:
def all_to_inds(pos, inds):
    dist_func = jax.vmap(lambda p1, p2: jnp.sqrt(jnp.sum((p2-p1)**2)), in_axes=(None, 1))
    distances = dist_func(pos, pos[:, inds, :])
    return distances

In [10]:
pos[:, traj.top.select('protein'), :].shape

(1, 5756, 3)

In [11]:
pos

array([[[ 6.6958,  1.538 , 10.0781],
        [ 6.5569,  1.5867, 10.0883],
        [ 6.5286,  1.6872,  9.9745],
        ...,
        [ 4.423 ,  3.3288,  8.2141],
        [ 4.8804,  3.4342,  8.5135],
        [ 3.9517,  3.6419,  8.3077]]], dtype=float32)

In [12]:
all_to_inds(pos, traj.top.select('protein'))

E1206 07:59:05.516854 2027348 cuda_executor.cc:1784] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
W1206 07:59:05.516988 2027348 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML library doesn't have required functions.
E1206 07:59:05.520218 2027279 cuda_executor.cc:1784] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
W1206 07:59:05.520277 2027279 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML library doesn't have required functions.
E1206 07:59:05.897334 2027279 gpu_hlo_schedule.cc:817] The byte size of input/output arguments (10462335840) exceeds the base limit (8049819648). This indicates an error in the calculation!
W1206 07:59:15.945353 2027279 bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.87GiB (rounded to 5231168000)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cud

JaxRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 4.87GiB.

**Prompt [Describe how to obtain relative free energies for... | Try in Copilot Chat](https://m365.cloud.microsoft/chat/entity1-d870f6cd-4aa5-4d42-9626-ab690c041429/eyJpZCI6IlZYTmxjbFl4ZkdoMGRIQnpPaTh2YzNWaWMzUnlZWFJsTFdsdWRDNXZabVpwWTJVdVkyOXRMM3hQU1VRNk5HSmxOR1JpT0RjdE1qSm1NQzAwT1dRd0xXSXhOalV0WlRGak5tVXlPV1JsTldVNWZHWTNabVF3WldVeUxXVmtOVGN0TkdVd1l5MDVNMlZpTFRFeE16WXpZalF3T0RSaE1ud3lNREkyTFRBeExUQTRWREl5T2pRNU9qSTFMamMyT1RnMk5qbGEiLCJzY2VuYXJpbyI6InNoYXJlTGlua1ZpYVJpY2hDb3B5IiwicHJvcGVydGllcyI6eyJwcm9tcHRTb3VyY2UiOiJ1c2VyIiwiY2xpY2tUaW1lc3RhbXAiOiIyMDI2LTAxLTA4VDIyOjQ5OjI2LjA2NVoifSwiY2hhdFR5cGUiOiJ3ZWIiLCJ2ZXJzaW9uIjoxLjF9)**
Describe how to obtain relative free energies for a set of ligand conformations, starting with the dPCA for the set of conformations up through performing metadynamics with native OpenMM.
**Response**
Great question, David. Here’s a **start‑to‑finish, practical workflow** to get **relative free energies** between a set of bound‑ligand conformations—starting from your **dihedral PCA (dPCA)** on an *ensemble* of conformations, and finishing with **well‑tempered metadynamics (WTMetaD) in native OpenMM**. I’ll focus on ligand‑internal DOFs (rotatable bonds) and keep the workflow reproducible and statistically sound.

---

## Overview (what you’ll do)

1) **Build dPCA** from your ensemble of bound conformations (no time correlation required).  
2) **Define CVs** (e.g., dPC1/dPC2) as trigonometric combinations of selected ligand torsions.  
3) **Implement CVs in OpenMM** via `CustomTorsionForce` + `CustomCVForce`.  
4) **Run WTMetaD (native OpenMM)** to enhance sampling across conformations.  
5) **Reconstruct free energy surface** \(F(\text{CV})\) from the accumulated bias.  
6) **Identify basins** (conformations) in CV space and compute **ΔG** between them.  
7) **Quantify uncertainty** (blocking/bootstrapping) and sanity‑check.

I’ll annotate with key references at each step.

---

## Step 1 — Build dPCA on ligand torsions (ensemble)

**What:** Select 3–10 **rotatable ligand torsions** that distinguish conformations, then perform PCA in **dihedral space** using the standard **sin/cos embedding** for periodicity:
\[
\phi_i \mapsto (\cos \phi_i, \; \sin \phi_i) \quad \Rightarrow \quad \text{PCA in } \mathbb{R}^{2N}
\]

**Why:** dPCA disentangles **internal** conformational variability and avoids rigid‑body/periodicity artifacts, giving you **interpretable collective modes** (dPC1, dPC2, …). [1](https://pubs.aip.org/aip/jcp/article/126/24/244111/295934/Dihedral-angle-principal-component-analysis-of)[2](https://europepmc.org/article/MED/17614541)[3](https://manual.gromacs.org/current/reference-manual/analysis/dihedral-pca.html)

> **Note:** dPCA is purely geometric (variance‑focused), which is ideal for your **independent conformations** use‑case; TICA isn’t applicable without time correlations. [1](https://pubs.aip.org/aip/jcp/article/126/24/244111/295934/Dihedral-angle-principal-component-analysis-of)

---

## Step 2 — Turn dPCs into analytic CVs

From dPCA, you get coefficients \(\{a_i, b_i\}\) for dPC1 (and \(\{c_i, d_i\}\) for dPC2):

\[
\mathrm{dPC1}(\boldsymbol{\phi}) = \sum_{i=1}^N \big(a_i \cos \phi_i + b_i \sin \phi_i\big), \qquad
\mathrm{dPC2}(\boldsymbol{\phi}) = \sum_{i=1}^N \big(c_i \cos \phi_i + d_i \sin \phi_i\big)
\]

These **scalar CVs** are smooth, non‑periodic, and ideal for WTMetaD. (They implement the same trigonometric embedding advocated by dPCA theory.) [1](https://pubs.aip.org/aip/jcp/article/126/24/244111/295934/Dihedral-angle-principal-component-analysis-of)

---

## Step 3 — Implement CVs in native OpenMM

**Key idea:** Feed each torsion angle \(\phi_i\) to a `CustomCVForce`, then combine via trigonometric coefficients.

```python
from openmm import *
from openmm.app import *
from openmm.unit import *

# 1) Define torsion forces that output angle values (theta)
t1_force = CustomTorsionForce("theta"); t1_force.addTorsion(i1,i2,i3,i4, [])
t2_force = CustomTorsionForce("theta"); t2_force.addTorsion(j1,j2,j3,j4, [])
# ... add one per selected torsion ...
system.addForce(t1_force); system.addForce(t2_force)

# 2) Define dPC1 as a CustomCVForce (energy equals the CV scalar)
dpc1_force = CustomCVForce("a1*cos(t1)+b1*sin(t1) + a2*cos(t2)+b2*sin(t2)")
dpc1_force.addCollectiveVariable("t1", t1_force)
dpc1_force.addCollectiveVariable("t2", t2_force)
# set your coefficients:
dpc1_force.addGlobalParameter("a1", 0.45)
dpc1_force.addGlobalParameter("b1", 0.60)
dpc1_force.addGlobalParameter("a2", 0.30)
dpc1_force.addGlobalParameter("b2", -0.10)
system.addForce(dpc1_force)
```

The native **OpenMM metadynamics** API is built around `CustomCVForce` and supports **well‑tempered MetaD**, multi‑walker via shared bias files, and 1–3 CVs. [4](https://docs.openmm.org/7.5.0/api-python/generated/simtk.openmm.app.metadynamics.Metadynamics.html)

---

## Step 4 — Run well‑tempered metadynamics (WTMetaD)

WTMetaD deposits Gaussian hills whose height decays in time, ensuring **smooth convergence** and a bias that asymptotically encodes the **free energy**. [5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)

```python
from openmm.app import metadynamics

temperature = 300*kelvin
biasFactor = 12.0           # γ = (T+ΔT)/T   (common: 8–20)
height = 0.5*kilojoules_per_mole
frequency = 500             # strides in MD steps

# Define the variable domain and hill width (sigma) from your ensemble histogram
bv_dpc1 = metadynamics.BiasVariable(dpc1_force, min_val=-2.0, max_val=2.0, bias_width=0.10)

metaD = metadynamics.Metadynamics(system, [bv_dpc1],
                                   temperature, biasFactor,
                                   height, frequency,
                                   saveFrequency=5000, biasDir="bias_out")

# Run MD with metadynamics stepping:
metaD.step(simulation, 1_000_000)  # e.g. 1M steps
```

**OpenMM docs**: WTMetaD is implemented in `openmm.app.metadynamics.Metadynamics`; it creates a bias force and periodically saves and aggregates bias files (useful for multi‑walker or resuming runs). [4](https://docs.openmm.org/7.5.0/api-python/generated/simtk.openmm.app.metadynamics.Metadynamics.html)

---

## Step 5 — Reconstruct the free energy surface \(F(s)\) from the bias

For **well‑tempered** MetaD, the long‑time relation between the converged bias \(V(s)\) and the **true free energy** \(F(s)\) is:
\[
F(s) \;\approx\; -\frac{\gamma}{\gamma-1}\, V(s) \;+\; C
\quad\text{with}\quad \gamma = \frac{T+\Delta T}{T}
\]
(Equivalently, \(F(s) \approx -\tfrac{T+\Delta T}{\Delta T}\,V(s) + C\).) This is the **Barducci–Bussi–Parrinello** well‑tempered MetaD result. [5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)

**How to get \(V(s)\) in OpenMM:**  
OpenMM stores the bias as a **continuous function/table** over the CV domain (1D/2D) and updates it every `saveFrequency`. You can read the saved bias grids and apply the factor above to get \(F(s)\). Example reference implementation (PEastman’s metadynamics table code): the bias is represented by `Continuous1DFunction` / `Continuous2DFunction`. [6](https://web.stanford.edu/~peastman/metadynamics.py)

> If you prefer a ready‑made analysis helper, **UFEDMM** demonstrates bias→free‑energy reconstruction patterns and reweighting utilities compatible with OpenMM (though its focus is extended‑space dynamics, the analysis examples show WTMetaD bias handling). [7](https://github.com/craabreu/ufedmm)[8](https://craabreu.github.io/ufedmm/_modules/ufedmm/analysis.html)[9](https://sites.google.com/view/openmm-ufedmm/tutorials/ufed)

---

## Step 6 — Extract **relative free energies** between conformations

### A. Define basins in CV space
Use **density clustering** or **hand‑drawn polygon masks** in \(\text{dPC1}(\text{dPC2})\) space to delineate conformations (e.g., pose A, pose B, rotamer C).

### B. Compute \(\Delta G\) via **free‑energy minima**
From \(F(s)\), take local minima (or basin integrals). The **relative free energy** between basins \(i\) and \(j\) is:
\[
\Delta G_{ij} \;=\; F(s_j^\*) \;-\; F(s_i^\*)
\quad\text{or}\quad
\Delta G_{ij} \;=\; -k_BT \ln \frac{\int_{\mathcal{B}_j} e^{-F(s)/k_BT}\,ds}{\int_{\mathcal{B}_i} e^{-F(s)/k_BT}\,ds}
\]
where \(s^\*\) denotes the basin minimum and \(\mathcal{B}_i\) the basin region.

### C. Alternatively, compute \(\Delta G\) from **reweighted probabilities**
WTMetaD permits **equilibrium reweighting**: reconstruct the unbiased distribution \(P(s)\) from the biased samples using the deposited bias history (Bussi–Laio–Parrinello). Then:
\[
\Delta G_{ij} \;=\; -k_BT \ln \frac{P_j}{P_i}
\]
with \(P_i = \int_{\mathcal{B}_i} P(s)\,ds\). [5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)

*(If you go this route, the UFEDMM analysis examples show how to turn stored hill centers/heights into a free‑energy function and binned statistics.)* [8](https://craabreu.github.io/ufedmm/_modules/ufedmm/analysis.html)

---

## Step 7 — Uncertainty & convergence checks

- **Time blocking / batch reweighting:** split your MetaD run into blocks, recompute \(F(s)\) and \(\Delta G\) per block, then take mean ± s.e.m.  
- **Hill decay & bias saturation:** In WTMetaD, ensure hills are small at late times and the reconstructed \(F(s)\) stabilizes. [5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)  
- **Basin robustness:** resample your basin boundaries slightly and check \(\Delta G\) stability.  
- **Physical sanity:** monitor key **protein–ligand contacts** (e.g., β₂AR Asp113^3.32 salt bridge, Ser204/Ser207 H‑bonds) to confirm enhanced sampling remains chemically realistic.

---

## Minimal working example (tying it together)

Below is a concise template for **native OpenMM WTMetaD** with one dPCA CV; adapt indices/coefficients/ranges to your ligand.

```python
# --- Build system, topology, integrator (omitted for brevity) ---

# 1) Torsion angle forces -> theta values
tforces = []
for (a,b,c,d) in torsion_atom_quartets:  # your ligand torsions (0-based indices)
    tf = CustomTorsionForce("theta"); tf.addTorsion(a,b,c,d, [])
    system.addForce(tf); tforces.append(tf)

# 2) dPC1 CV force: sum a_i*cos(ti) + b_i*sin(ti)
expr_terms = []
for k, (ai, bi) in enumerate(zip(a_coeffs, b_coeffs), start=1):
    expr_terms += [f"a{k}*cos(t{k})", f"b{k}*sin(t{k})"]
dpc1_force = CustomCVForce(" + ".join(expr_terms))
for k, tf in enumerate(tforces, start=1):
    dpc1_force.addCollectiveVariable(f"t{k}", tf)
for k, (ai, bi) in enumerate(zip(a_coeffs, b_coeffs), start=1):
    dpc1_force.addGlobalParameter(f"a{k}", ai)
    dpc1_force.addGlobalParameter(f"b{k}", bi)
system.addForce(dpc1_force)

# 3) WTMetaD setup (OpenMM native)
from openmm.app import metadynamics
bv = metadynamics.BiasVariable(dpc1_force, min_val=-2.0, max_val=2.0, bias_width=0.10)
metaD = metadynamics.Metadynamics(system, [bv],
                                  temperature=300*kelvin, biasFactor=12.0,
                                  height=0.5*kilojoules_per_mole,
                                  frequency=500, saveFrequency=5000, biasDir="bias_out")

# 4) Run and save bias
metaD.step(simulation, 1_000_000)

# 5) Post-process: read saved bias grid and compute F(s) = -γ/(γ-1) * V(s) + C
gamma = 12.0
scale = -gamma/(gamma-1)  # apply to bias to get free energy (up to constant)
# Load grid(s) from bias_out and produce F(s) curve; then compute ΔG between basins.
```

**Why this works:** OpenMM’s class implements WTMetaD and documents using the bias to compute \(F(s)\) in CV space; applying the **Barducci et al.** scaling yields free energies (up to a constant). [4](https://docs.openmm.org/7.5.0/api-python/generated/simtk.openmm.app.metadynamics.Metadynamics.html)[5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)

If you prefer a reference implementation of the **bias grid**, Stanford’s `metadynamics.py` shows a table‑based bias updated during the run; you can mirror that logic when reading the saved grids. [6](https://web.stanford.edu/~peastman/metadynamics.py)

---

## Practical parameter tips (ligand in β₂AR)

- **Sigma (bias width):** estimate from the **ensemble** standard deviation of dPC1; start with ~0.8–1.2× that value.  
- **Height:** conservative (e.g., 0.3–0.5 kJ/mol per hill) to avoid ejecting the ligand from the pocket; increase only if sampling is stuck.  
- **Bias factor γ:** 8–20; larger γ speeds barrier flattening but can over‑explore.  
- **Domain [min, max]:** take observed dPC range ±10–20% margin from pre‑MetaD ensemble.

All are consistent with **WTMetaD best practices** and OpenMM’s built‑in MetaD documentation. [4](https://docs.openmm.org/7.5.0/api-python/generated/simtk.openmm.app.metadynamics.Metadynamics.html)[5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)

---

## Common pitfalls (and fixes)

- **Unphysical excursions:** add simple **restraints/walls** on auxiliary CVs (e.g., ligand COM distance from key residues) using OpenMM custom forces; if you later need richer walls/grids, consider PLUMED (with some speed tradeoff). [10](https://ash.readthedocs.io/en/latest/Biased-sampling.html)  
- **Poor basin separability:** revisit torsion selection; dPCs must reflect **chemically meaningful rotations** (back to Step 1). [1](https://pubs.aip.org/aip/jcp/article/126/24/244111/295934/Dihedral-angle-principal-component-analysis-of)  
- **Bias not converged:** extend run or reduce hill height; WTMetaD converges smoothly when hill heights decay and the bias stops growing in visited basins. [5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)

---

## Wrap‑up

- Use **ligand‑only dPCA** to define **interpretable CVs**. [1](https://pubs.aip.org/aip/jcp/article/126/24/244111/295934/Dihedral-angle-principal-component-analysis-of)  
- Run **native OpenMM WTMetaD** on those CVs (fastest path). [4](https://docs.openmm.org/7.5.0/api-python/generated/simtk.openmm.app.metadynamics.Metadynamics.html)  
- Reconstruct **\(F(s)\)** from the accumulated bias using the **well‑tempered scaling**; extract **ΔG** between basins. [5](https://www.ks.uiuc.edu/Research/namd/2.9/ug/node97.html)[6](https://web.stanford.edu/~peastman/metadynamics.py)

If you share your **ligand torsion quartets** and **dPCA coefficients**, I can generate a *ready‑to‑run* `CustomCVForce` snippet and a small post‑processing script that reads OpenMM’s saved bias and outputs **ΔG** with uncertainties.